You can now transform labeled tables with pandas and reason about NumPy shapes. This chapter puts those skills together: **use pandas to prepare and interpret data, and use NumPy where an array calculation improves the workflow.**

## Learning Objectives

- Identify a numerical operation worth optimizing.
- Convert selected pandas columns to arrays without losing track of records.
- Replace row functions and nested loops with array operations.
- Restore meaningful labels to computed results.
- Check correctness before measuring speed, including conversion costs.

## The Data Science Workflow

1. **Load:** read data with pandas.
2. **Prepare:** inspect types, missing values, identifiers, and units; select relevant records.
3. **Align:** explicitly order the numeric inputs and retain labels.
4. **Compute:** use a suitable NumPy operation on the arrays.
5. **Return:** build labeled Series or DataFrames from results.
6. **Validate:** compare against a correct reference, including missing values and labels.
7. **Interpret and export:** summarize, plot, or save using pandas.

Conversion is a decision within the workflow, not a required step for every calculation. Start with a clear pandas solution. The worked shopping example below carries one problem through all seven steps.


In [1]:
import numpy as np
import pandas as pd
from timeit import repeat
from pathlib import Path
from tempfile import TemporaryDirectory
print('pandas', pd.__version__, 'NumPy', np.__version__)


pandas 3.0.5 NumPy 2.5.3


## Start with a Correct Pandas Solution

A direct column expression is already an effective way to calculate revenue, discounts, or summaries. Replacing a Python row function with a column expression can matter more than switching libraries. Use `map` for lookups and pandas for labels, strings, dates, joins, and grouping.

Vectorization expresses an operation over arrays. Numerical routines often execute loops in compiled code; vectorization does **not** mean that every operation runs in parallel. Neither a batch-looking API nor a conversion guarantees a speedup.


In [2]:
orders = pd.DataFrame({'price': [20., 50., 30.], 'quantity': [2, 1, 3],
                       'discount': [0.1, 0.2, 0.]}, index=[104, 101, 109])
def row_total(row):
    return row['price'] * row['quantity'] * (1 - row['discount'])
reference = orders.apply(row_total, axis=1)
pandas_total = orders['price'] * orders['quantity'] * (1 - orders['discount'])
pd.testing.assert_series_equal(reference, pandas_total)
print(pandas_total)


104    36.0
101    40.0
109    90.0
dtype: float64


## Convert Selected Values, Preserve Their Meaning

Select columns **by name and in an explicit order**. Keep the index with the exact rows being converted. A Series becomes a 1D array; a DataFrame becomes a 2D array. Arrays carry positions rather than pandas labels.

`to_numpy(dtype=..., na_value=...)` controls the common array dtype and missing-value representation. A mixed table can become an object array, unsuitable for many fast numerical operations. `copy=False` does not guarantee zero copying; use `copy=True` when you need an independent writable array. Do not assume a returned array can safely be edited in place. See [pandas conversion documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_numpy.html).


In [3]:
selected = orders.loc[:, ['price', 'quantity', 'discount']]
saved_index = selected.index.copy()
values = selected.to_numpy(dtype=float, copy=True)
print('Table:', selected.shape, 'Array:', values.shape, values.dtype)
print('One column:', selected['price'].to_numpy().shape)
price, quantity, discount = values[:, 0], values[:, 1], values[:, 2]
array_total = price * quantity * (1 - discount)
result = pd.Series(array_total, index=saved_index, name='total')
pd.testing.assert_series_equal(result, pandas_total, check_names=False)
print(result)


Table: (3, 3) Array: (3, 3) float64
One column: (3,)
104    36.0
101    40.0
109    90.0
Name: total, dtype: float64


### Missing Values and Statistical Conventions

Establish a policy before conversion. Here a missing measurement stays missing. Use matching `ddof` settings when comparing standard deviations: pandas and NumPy defaults differ. A constant column has zero standard deviation, so its z-score is undefined under this formula.


In [4]:
measurements = pd.DataFrame({'x': pd.array([1, 2, None, 4], dtype='Float64'),
                             'constant': [5., 5., 5., 5.]}, index=[8, 3, 12, 7])
a = measurements.to_numpy(dtype=float, na_value=np.nan, copy=True)
means = np.nanmean(a, axis=0)
scales = np.nanstd(a, axis=0, ddof=1)
z = np.full(a.shape, np.nan)
np.divide(a - means, scales, out=z, where=scales != 0)
z_numpy = pd.DataFrame(z, index=measurements.index, columns=measurements.columns)
numeric_measurements = measurements.astype('float64')
z_pandas = (numeric_measurements - numeric_measurements.mean()) / numeric_measurements.std(ddof=1).replace(0, float('nan'))
np.testing.assert_allclose(z_numpy.to_numpy(), z_pandas.to_numpy(dtype=float, na_value=np.nan), equal_nan=True)
print(z_numpy)


           x  constant
8  -0.872872       NaN
3  -0.218218       NaN
12       NaN       NaN
7   1.091089       NaN


### Return Results to the Correct Records

Assigning an array uses current row positions. Assigning a labeled Series aligns by its index. If you sort or filter the table after conversion, a bare result array can silently attach values to the wrong records. Keep a unique identifier, and preserve or explicitly transform its ordering alongside the array. If labels are duplicated, establish a unique row key first.


In [5]:
sorted_orders = orders.sort_index().copy()
sorted_orders['wrong'] = array_total  # Demonstration of an incorrect positional assignment
sorted_orders['correct'] = result    # Labels align despite sorting
print(sorted_orders[['wrong', 'correct']])
assert not sorted_orders['wrong'].equals(sorted_orders['correct'])
pd.testing.assert_series_equal(sorted_orders['correct'], result.sort_index(), check_names=False)


     wrong  correct
101   36.0     40.0
104   40.0     36.0
109   90.0     90.0


## Worked Workflow: Shopping Costs {#shopping-workflow}

Three shoppers need four products, and two stores offer different prices. Which store minimizes each shopper's bill? The files are the same ones used in Pandas Intermediate.

### Load, Inspect, and Align

The quantities have **people in rows and products in columns**. Prices need **products in rows and stores in columns**. Align the shared product dimension by labels before dropping those labels. Missing prices must not be treated as free items.


In [6]:
food = pd.read_csv('Datasets/food_quantity.csv').set_index('Person')
raw_prices = pd.read_csv('Datasets/price.csv').set_index('Item')
items = ['roll', 'bun', 'cake', 'bread']
stores = ['Target', 'Kroger']
assert food.index.is_unique and raw_prices.index.is_unique
quantities = food.loc[:, items].apply(pd.to_numeric, errors='raise')
prices = raw_prices.loc[items, stores].apply(pd.to_numeric, errors='raise')
assert not quantities.isna().any().any() and not prices.isna().any().any()
assert np.isfinite(quantities.to_numpy()).all() and np.isfinite(prices.to_numpy()).all()
assert (quantities >= 0).all().all() and (prices >= 0).all().all()
assert quantities.columns.equals(prices.index)
print(quantities)
print(prices)


         roll  bun  cake  bread
Person                         
Ben         6    5     3      1
Barbara     3    6     2      2
Beth        3    4     3      1
       Target  Kroger
Item                 
roll      1.5     1.0
bun       2.0     2.5
cake      5.0     4.5
bread    16.0    17.0


### Establish Reference Results

Use a small explicit loop to make the sum of quantity × price visible. Also retain a pandas solution that multiplies by product labels. These are independent checks on the array result.


In [7]:
def loop_costs(q, p):
    totals = []
    for person in q.index:
        person_totals = []
        for store in p.columns:
            total = 0.0
            for item in q.columns:
                total += q.loc[person, item] * p.loc[item, store]
            person_totals.append(total)
        totals.append(person_totals)
    return pd.DataFrame(totals, index=q.index, columns=p.columns)

def pandas_costs(q, p):
    return pd.DataFrame({store: q.mul(p[store], axis='columns').sum(axis=1)
                         for store in p.columns}, index=q.index)

reference_costs = loop_costs(quantities, prices)
pd.testing.assert_frame_equal(reference_costs, pandas_costs(quantities, prices))
print(reference_costs)


         Target  Kroger
Person                 
Ben        50.0    49.0
Barbara    58.5    61.0
Beth       43.5    43.5


### Convert and Compute with Matrix Multiplication

`Q @ P` multiplies each shopper's quantities by a store's prices and sums across products. Its dimensions are `(3, 4) @ (4, 2) → (3, 2)`: people × stores.

`*` is element-wise multiplication; `@` is matrix multiplication. Do not reshape an unrelated array merely to make dimensions fit. The shared dimension must describe the same products in the same order.


In [8]:
Q = quantities.to_numpy(dtype=float)
P = prices.to_numpy(dtype=float)
assert Q.shape[1] == P.shape[0]
all_costs = Q @ P
print('Shapes:', Q.shape, '@', P.shape, '->', all_costs.shape)
print('Ben at Target:', Q[0] @ P[:, 0])
print('Ben at both stores:', Q[0] @ P)


Shapes: (3, 4) @ (4, 2) -> (3, 2)
Ben at Target: 50.0
Ben at both stores: [50. 49.]


### Restore Labels, Validate, and Interpret

The output columns are stores, not the original product columns. Restore labels according to the **meaning of the result**. Report all tied cheapest stores rather than allowing the first position to decide a tie silently.


In [9]:
costs = pd.DataFrame(all_costs, index=quantities.index.copy(), columns=prices.columns.copy())
pd.testing.assert_frame_equal(costs, reference_costs)
best = costs.min(axis=1)
is_best = costs.eq(best, axis='index')
report = costs.copy()
report['cheapest_stores'] = is_best.apply(lambda row: ', '.join(row.index[row]), axis=1)
report['savings_vs_other_store'] = costs.max(axis=1) - best
print(report)
# An isolated export demonstrates the last workflow step without overwriting a user's file.
with TemporaryDirectory() as folder:
    output = Path(folder) / 'shopping_report.csv'
    report.to_csv(output)
    loaded_report = pd.read_csv(output, index_col=0)
    pd.testing.assert_frame_equal(report, loaded_report)


         Target  Kroger cheapest_stores  savings_vs_other_store
Person                                                         
Ben        50.0    49.0          Kroger                     1.0
Barbara    58.5    61.0          Target                     2.5
Beth       43.5    43.5  Target, Kroger                     0.0


### A Shape Check Is Not Enough

Reverse the price rows. The dimensions still fit, but unaligned multiplication pairs products incorrectly. Reindexing the price table repairs the meaning before conversion.


In [10]:
shuffled_prices = prices.iloc[::-1]
wrong_costs = Q @ shuffled_prices.to_numpy(dtype=float)
assert not np.allclose(wrong_costs, all_costs)
aligned_prices = shuffled_prices.loc[quantities.columns]
np.testing.assert_allclose(Q @ aligned_prices.to_numpy(dtype=float), all_costs)
print('Realigning product labels restores the correct bills.')


Realigning product labels restores the correct bills.


## Replace Row Functions with Conditional Arrays

Some NumPy functions can take pandas Series directly, without converting an entire table. `np.select` returns an array and uses the **first true condition**, so order rules from highest priority to lowest. Define a missing-input policy explicitly. This example contains no missing values; the checks make that assumption visible.


In [11]:
customers = pd.DataFrame({'amount': [700., 150., 300., 80., 600.],
                          'premium': [True, True, False, False, False],
                          'rating': [5, 4, 4, 2, 3]}, index=[41, 12, 73, 5, 62])
assert not customers.isna().any().any()
def segment_row(row):
    if row['premium'] and row['amount'] > 500:
        return 'VIP'
    if row['premium'] and row['rating'] >= 4:
        return 'Premium+'
    if row['amount'] > 200 and row['rating'] >= 4:
        return 'High Value'
    return 'Standard'
def segment_numpy(frame):
    conditions = [frame['premium'] & (frame['amount'] > 500),
                  frame['premium'] & (frame['rating'] >= 4),
                  (frame['amount'] > 200) & (frame['rating'] >= 4)]
    return pd.Series(np.select(conditions, ['VIP', 'Premium+', 'High Value'], default='Standard'),
                     index=frame.index, name='segment')
segments = segment_numpy(customers)
pd.testing.assert_series_equal(segments, customers.apply(segment_row, axis=1), check_names=False)
print(customers.assign(segment=segments))


    amount  premium  rating     segment
41   700.0     True       5         VIP
12   150.0     True       4    Premium+
73   300.0    False       4  High Value
5     80.0    False       2    Standard
62   600.0    False       3    Standard


### Numerical Functions and Broadcasting

Use numerical functions for transformations and broadcasting for one adjustment per row or column. This example has observations in rows and features in columns. A per-row divisor needs shape `(n, 1)`; a per-feature weight needs shape `(p,)`. Guard zero denominators before division. `np.vectorize` is a convenience wrapper around Python calls, not the optimization illustrated here.


In [12]:
features = pd.DataFrame({'visits': [2., 5., 0.], 'purchases': [1., 3., 0.]}, index=[104, 101, 109])
X = features.to_numpy(dtype=float)
weights = np.array([0.25, 0.75])
weighted = X * weights
row_totals = X.sum(axis=1, keepdims=True)
shares = np.full(X.shape, np.nan)
np.divide(X, row_totals, out=shares, where=row_totals != 0)
print(pd.DataFrame(weighted, index=features.index, columns=features.columns))
print(pd.DataFrame(shares, index=features.index, columns=features.columns))
print('Log-transformed visits:', np.log1p(features['visits']))


     visits  purchases
104    0.50       0.75
101    1.25       2.25
109    0.00       0.00
       visits  purchases
104  0.666667   0.333333
101  0.625000   0.375000
109       NaN        NaN
Log-transformed visits: 104    1.098612
101    1.791759
109    0.000000
Name: visits, dtype: float64


## Measure the Whole Workflow

First verify matching outputs. Then repeat timings on modest and larger inputs. Exclude data generation and printing from timing. Report **array computation alone** separately from **conversion + computation + labeled reconstruction**; only the latter is comparable to a full pandas operation.

These repeated shopper records are synthetic benchmark inputs, not additional real observations. Library versions, data size, dtypes, and hardware affect timing. Small examples teach correctness but may not benefit from conversion.


In [13]:
def numpy_costs(q, p):
    return pd.DataFrame(q.to_numpy(dtype=float) @ p.to_numpy(dtype=float),
                        index=q.index, columns=p.columns)
benchmark_rows = []
for count in [3, 3000, 30000]:
    q = pd.concat([quantities] * (count // len(quantities)), ignore_index=True)
    qa = q.to_numpy(dtype=float)
    pa = prices.to_numpy(dtype=float)
    expected = pandas_costs(q, prices)
    pd.testing.assert_frame_equal(numpy_costs(q, prices), expected)
    np.testing.assert_allclose(qa @ pa, expected.to_numpy())
    operations = {'pandas': lambda: pandas_costs(q, prices),
                  'numpy round trip': lambda: numpy_costs(q, prices),
                  'array computation only': lambda: qa @ pa}
    for label, operation in operations.items():
        seconds = min(repeat(operation, repeat=5, number=3)) / 3
        benchmark_rows.append({'rows': count, 'method': label, 'seconds': seconds})
benchmark = pd.DataFrame(benchmark_rows)
print(benchmark.to_string(index=False))
comparison = benchmark.pivot(index='rows', columns='method', values='seconds')
comparison['observed pandas / numpy round-trip time'] = comparison['pandas'] / comparison['numpy round trip']
print(comparison)


 rows                 method      seconds
    3                 pandas 2.619860e-04
    3       numpy round trip 5.430348e-06
    3 array computation only 3.886720e-07
 3000                 pandas 4.393053e-04
 3000       numpy round trip 2.037501e-05
 3000 array computation only 7.791639e-06
30000                 pandas 2.144944e-03
30000       numpy round trip 1.073056e-04
30000 array computation only 6.375000e-05
method  array computation only  ...  observed pandas / numpy round-trip time
rows                            ...                                         
3                 3.886720e-07  ...                                48.244792
3000              7.791639e-06  ...                                21.560987
30000             6.375000e-05  ...                                19.989113

[3 rows x 4 columns]


The final ratio exceeds 1 when the measured NumPy round trip is faster. A value below 1 means pandas was faster for that measurement. Do not generalize the shopping result to every operation.

Now measure the conditional example against its row-function reference. Both methods construct a labeled result, and neither mutates the input.


In [14]:
larger_customers = pd.concat([customers] * 1000, ignore_index=True)
pd.testing.assert_series_equal(segment_numpy(larger_customers),
                               larger_customers.apply(segment_row, axis=1), check_names=False)
for label, operation in [('row apply', lambda: larger_customers.apply(segment_row, axis=1)),
                         ('conditional arrays', lambda: segment_numpy(larger_customers))]:
    print(label, min(repeat(operation, repeat=3, number=1)), 'seconds')


row apply 0.011287542060017586 seconds
conditional arrays 0.000330875045619905 seconds


## Independent Practice: Movie Ratings

Use `Datasets/movies_cleaned.csv`. Genre flags are `comedy`, `Action`, `drama`, and `horror`; the rating columns are `IMDB Rating` and `Rotten Tomatoes Rating`. A movie may belong to more than one genre. These ratings describe this dataset, not all viewers' preferences.

1. Inspect types and missingness. For the first solution, use the same complete cases for both rating columns and all genre flags; report how many records you exclude.
2. Create a pandas reference: for each genre, select movies with flag 1 and calculate the two mean ratings.
3. Build `R` with shape `(movies, 2)` and `G` with shape `(movies, 4)`, keeping exactly the same row order.
4. Compute totals with `R.T @ G`, counts with `G.sum(axis=0)`, and means by dividing totals by counts. Leave a genre with zero movies undefined.
5. Return a DataFrame with rating names as rows and genres as columns. Check it against the pandas reference, including labels and missing values.
6. Report the results separately for each rating scale. Measure the complete pandas and NumPy workflows; explain whether conversion helped.
7. Deliberately shuffle only one input, demonstrate the problem, and repair it using record identifiers before conversion.

**Extension:** Retain partially rated movies. Work out a separate observed-rating denominator for every rating–genre pair instead of using one count for all ratings.

## Optional Extension: Random Generation, Simulation, and Bootstrapping

Use a local generator for reproducible numerical examples. The [NumPy random guide](https://numpy.org/doc/stable/reference/random/index.html) explains generators and distributions. A seed makes a run repeatable in a given environment; it does not make simulated observations real data.


In [15]:
rng = np.random.default_rng(303)
print('Uniform:', rng.uniform(5, 25, size=5))
print('Normal:', rng.normal(8, 3, size=5))
print('Integers:', rng.integers(1, 7, size=5))
print('Without replacement:', rng.choice(['A', 'B', 'C'], size=2, replace=False))
print('Binomial:', rng.binomial(10, 0.5, size=5))
print('Poisson:', rng.poisson(3, size=5))
print('Exponential:', rng.exponential(5, size=5))
print('Beta:', rng.beta(2, 5, size=5))


Uniform: [ 9.28864838 13.33643642 21.1539048  10.47846552 21.31559063]
Normal: [12.87278602  4.85449395  4.97015823  4.04806822  9.20514635]
Integers: [1 3 5 2 5]
Without replacement: ['B' 'C']
Binomial: [7 5 4 4 8]
Poisson: [1 3 2 0 5]
Exponential: [ 1.01543685 10.09064987  2.79028265 16.55579795 25.39823656]
Beta: [0.35435286 0.21817798 0.23217628 0.1663974  0.35372166]


### Simulation Practice

Simulate waiting times for 500 customers on each of 30 days. Use a uniform distribution from 5 to 25 minutes for cart 1 and a nonnegative distribution of your choice for cart 2. Explain its parameters and whether paired customer comparisons are meaningful. A normal distribution can produce negative times, so it is not automatically a suitable waiting-time model.

Generate arrays with shape `(30, 500)`. Calculate average waiting times per day, the number of days cart 2 has the larger mean, and the fraction of paired observations where its wait is longer. Validate your reductions with a loop over the **same generated arrays**, then time the computations separately from generation.

### Bootstrap Workflow: From Movies to Arrays and Back

Use worldwide gross minus production budget as a **gross-minus-budget measure**, not accounting profit: other costs and revenue shares are not supplied. Filter complete finite observations with pandas, resample a NumPy array, and return a summary. This percentile interval describes uncertainty under the resampling assumptions; it does not correct selection bias in the dataset.


In [16]:
movies = pd.read_csv('Datasets/movies_cleaned.csv')
inputs = movies.loc[movies['Action'].eq(1), ['Worldwide Gross', 'Production Budget']].apply(pd.to_numeric, errors='coerce')
valid = inputs.notna().all(axis=1) & np.isfinite(inputs.to_numpy(dtype=float)).all(axis=1)
inputs = inputs.loc[valid]
gross_minus_budget = inputs['Worldwide Gross'] - inputs['Production Budget']
v = gross_minus_budget.to_numpy(dtype=float)
assert len(v) > 1
bootstrap_rng = np.random.default_rng(303)
# A modest 1000 x n sample matrix; use batches when n is large.
samples = bootstrap_rng.choice(v, size=(1000, len(v)), replace=True)
bootstrap_means = samples.mean(axis=1)
low, high = np.percentile(bootstrap_means, [2.5, 97.5])
bootstrap_report = pd.DataFrame({'n': [len(v)], 'mean': [v.mean()], 'lower_95': [low], 'upper_95': [high]},
                                index=['Action: gross minus budget'])
print(bootstrap_report)


                              n          mean      lower_95      upper_95
Action: gross minus budget  264  1.567146e+08  1.334189e+08  1.829125e+08


## Before You Move On

For any proposed speedup, explain: What are the axes? Which records and columns were converted? What happened to missing values and labels? Does the result match a reference? Does the complete workflow actually run faster?

Use pandas when it expresses the task clearly. Introduce NumPy when its array operations help, preserve meaning at both conversion boundaries, and measure the benefit. Continue to [Data Visualization](Data%20visualization.ipynb).
